# 06 Robustness across test-pair cohorts

This notebook applies the **same fixed model, evaluator, class pair, metrics and baseline definitions** as notebook 05 to 3 additional reproducible cohorts of official-test source/target images. It asks whether the observed path behavior is stable across sampled image pairs.
Note: it does not select a better checkpoint.

The fixed model was trained specifically for Sneaker (7) -> Ankle Boot (9). Changing class pairs without training the corresponding task model would test out-of-scope generalization, so this robustness check varies the test cohort and seed only.

In [ ]:
from pathlib import Path
import csv
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
repo_candidates = [cwd, cwd / 'minRF_interpolation']
repo_candidates.extend(
    candidate
    for parent in cwd.parents
    for candidate in (parent, parent / 'minRF_interpolation')
)
repo_candidates.extend((Path('/content/minRF_interpolation'), Path('/content/minrf/minRF_interpolation')))
repo_root = next(
    (candidate for candidate in repo_candidates if (candidate / 'api' / 'flow_model_api.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Repository not found. Run notebooks/00_colab_setup.ipynb first.')
sys.path.insert(0, str(repo_root))

from notebooks.utils.colab_paths import find_repository_root

repo_root = find_repository_root(repo_root)
os.chdir(repo_root)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SOURCE_LABEL, TARGET_LABEL = 7, 9
N_PAIRS, STEPS = 64, 50
COHORT_SEEDS = (2027, 2028, 2029)
RESULTS_DIR = repo_root / 'assets' / 'evaluation'
PROTOCOL_PATH = RESULTS_DIR / 'evaluation_protocol.json'
EVALUATOR_PATH = RESULTS_DIR / 'fmnist_eval_classifier_v1.pt'
if not PROTOCOL_PATH.is_file() or not EVALUATOR_PATH.is_file():
    raise FileNotFoundError('Run notebooks/04_evaluation_setup.ipynb first.')
protocol = json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))
if protocol['flow_pair'] != {'source': SOURCE_LABEL, 'target': TARGET_LABEL}:
    raise ValueError('Notebook pair does not match the frozen protocol')
if (protocol['evaluation_pairs'], protocol['trajectory_steps']) != (N_PAIRS, STEPS):
    raise ValueError('Notebook evaluation settings do not match the frozen protocol')
print(f'Robustness cohorts: {COHORT_SEEDS}; {N_PAIRS} pairs per cohort; device: {device}')

In [ ]:
from api.evaluation_api import load_eval_classifier
from api.flow_model_api import DatasetSpec, build_dataset_loaders
from notebooks.utils.evaluation_workflow import (
    METHOD_ORDER,
    bootstrap_mean_interval,
    build_reference_banks,
    evaluate_class_pair_cohort,
    load_fixed_flow,
)

checkpoint_path = repo_root / protocol['flow_checkpoint']
flow_model, checkpoint_metadata, checkpoint_config = load_fixed_flow(
    checkpoint_path,
    device,
    expected_source=SOURCE_LABEL,
    expected_target=TARGET_LABEL,
    expected_profile='medium',
)
evaluator = load_eval_classifier(EVALUATOR_PATH, device)
data = build_dataset_loaders(
    DatasetSpec(
        root=str(repo_root / 'data'),
        source_class=SOURCE_LABEL,
        target_class=TARGET_LABEL,
        seed=42,
    ),
    download=True,
)
reference_banks = build_reference_banks(
    data['train_dataset'],
    data['train_label_index'],
    (SOURCE_LABEL, TARGET_LABEL),
)

In [ ]:
cohorts = {}
for seed in COHORT_SEEDS:
    cohorts[seed] = evaluate_class_pair_cohort(
        flow_model=flow_model,
        eval_model=evaluator,
        test_dataset=data['test_dataset'],
        test_label_index=data['test_label_index'],
        reference_banks=reference_banks,
        source_label=SOURCE_LABEL,
        target_label=TARGET_LABEL,
        n_pairs=N_PAIRS,
        steps=STEPS,
        seed=seed,
        device=device,
    )
print(f'Completed {len(cohorts)} reproducible cohorts using the same RF checkpoint.')

## Cohort-to-cohort stability

Lines show the mean target-confidence and frozen-evaluator feature curves within each cohort. Shaded regions show **between-cohort standard deviation**, not a confidence interval. This makes seed-to-seed cohort variation visible without treating potentially overlapping test images as independent replications. Feature distances use normalized evaluator embeddings and training-split references; they are representation-dependent proxies, not realism guarantees.

In [ ]:
fig, axes = plt.subplots(1, len(METHOD_ORDER), figsize=(19, 4), sharey=True)
time = np.linspace(0, 1, STEPS + 1)
for axis, method in zip(axes, METHOD_ORDER):
    cohort_curves = []
    for seed, cohort in cohorts.items():
        samples = cohort['methods'][method]['metrics']['sample_metrics']
        cohort_curves.append(np.mean([s['target_confidence'] for s in samples], axis=0))
        axis.plot(time, cohort_curves[-1], alpha=0.55, label=f'seed {seed}')
    cohort_curves = np.asarray(cohort_curves)
    mean = cohort_curves.mean(axis=0)
    spread = cohort_curves.std(axis=0)
    axis.plot(time, mean, color='black', linewidth=2, label='cohort mean')
    axis.fill_between(time, mean - spread, mean + spread, color='black', alpha=0.12)
    axis.set_title(method.replace('_', ' '))
    axis.set_xlabel('t')
    axis.grid(alpha=0.2)
axes[0].set_ylabel('Mean P(target | xₜ)')
axes[-1].legend(fontsize=7)
fig.suptitle('Target-class progression across independent seed-defined test cohorts')
fig.tight_layout()
robustness_figure = RESULTS_DIR / 'robustness_target_confidence.png'
fig.savefig(robustness_figure, dpi=160, bbox_inches='tight')
plt.show()

feature_specs = [
    ('target_reference_distance', 'Nearest target-reference cosine distance'),
    ('pair_reference_distance', 'Nearest source/target support distance'),
    ('source_feature_similarity', 'Similarity to source embedding'),
    ('feature_step_distance', 'Local feature step distance'),
]
feature_fig, feature_axes = plt.subplots(
    len(feature_specs), len(METHOD_ORDER), figsize=(19, 12), squeeze=False
)
for row_index, (metric, label) in enumerate(feature_specs):
    for column_index, method in enumerate(METHOD_ORDER):
        axis = feature_axes[row_index, column_index]
        times = (np.arange(STEPS) + 0.5) / STEPS if metric == 'feature_step_distance' else time
        cohort_curves = []
        for seed, cohort in cohorts.items():
            samples = cohort['methods'][method]['feature_metrics']['sample_metrics']
            curve = np.mean([sample[metric] for sample in samples], axis=0)
            cohort_curves.append(curve)
            axis.plot(times, curve, alpha=0.5, linewidth=1, label=f'seed {seed}')
        cohort_curves = np.asarray(cohort_curves)
        mean = cohort_curves.mean(axis=0)
        spread = cohort_curves.std(axis=0)
        axis.plot(times, mean, color='black', linewidth=1.7, label='cohort mean')
        axis.fill_between(times, mean - spread, mean + spread, color='black', alpha=0.1)
        if row_index == 0:
            axis.set_title(method.replace('_', ' '))
        if column_index == 0:
            axis.set_ylabel(label)
        axis.set_xlabel('t')
        axis.grid(alpha=0.2)
feature_axes[0, -1].legend(fontsize=6)
feature_fig.suptitle('Frozen-evaluator feature metrics across seeded test cohorts')
feature_fig.tight_layout()
feature_figure = RESULTS_DIR / 'robustness_feature_curves.png'
feature_fig.savefig(feature_figure, dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
rows = []
for seed, cohort in cohorts.items():
    for method in METHOD_ORDER:
        method_result = cohort['methods'][method]
        samples = method_result['metrics']['sample_metrics']
        pixel_steps = np.asarray(method_result['pixel_smoothness']['per_sample_step_sizes']).T
        feature_samples = method_result['feature_metrics']['sample_metrics']
        rows.append({
            'seed': seed,
            'method': method,
            'target Δ (end-start)': float(np.mean([
                s['target_confidence'][-1] - s['target_confidence'][0] for s in samples
            ])),
            'monotonic paths (%)': 100 * np.mean([
                s['target_confidence_monotonic'] for s in samples
            ]),
            'source confidence drop': float(np.mean([
                s['source_confidence_drop'] for s in samples
            ])),
            'probability step change': float(np.mean([
                np.mean(s['prob_gradient']) for s in samples
            ])),
            'pixel step norm': float(pixel_steps.mean()),
            'target feature distance Δ': float(np.mean([
                s['target_reference_distance'][-1] - s['target_reference_distance'][0]
                for s in feature_samples
            ])),
            'source feature similarity (end)': float(np.mean([
                s['source_feature_similarity'][-1] for s in feature_samples
            ])),
            'pair support distance (mean)': float(np.mean([
                np.mean(s['pair_reference_distance']) for s in feature_samples
            ])),
            'feature step distance (mean)': float(np.mean([
                np.mean(s['feature_step_distance']) for s in feature_samples
            ])),
        })

header = '| Seed | Method | Target Δ | Monotonic paths (%) | Source confidence drop | Probability step change | Pixel step norm | Target feature distance Δ | Source feature similarity (end) | Pair support distance (mean) | Feature step distance (mean) |\n|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|\n'
body = ''.join(
    f"| {row['seed']} | {row['method']} | {row['target Δ (end-start)']:.3f} | "
    f"{row['monotonic paths (%)']:.1f} | {row['source confidence drop']:.3f} | "
    f"{row['probability step change']:.4f} | {row['pixel step norm']:.3f} | {row['target feature distance Δ']:.3f} | "
    f"{row['source feature similarity (end)']:.3f} | {row['pair support distance (mean)']:.3f} | {row['feature step distance (mean)']:.3f} |\n"
    for row in rows
)
display(Markdown(header + body + '\nThese are cohort summaries, not model-selection scores.'))

csv_path = RESULTS_DIR / 'robustness_cohort_summary.csv'
with csv_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
result = {
    'protocol': protocol,
    'checkpoint_metadata': checkpoint_metadata,
    'checkpoint_config': checkpoint_config,
    'cohort_seeds': list(COHORT_SEEDS),
    'cohorts': {
        str(seed): {
            'config': cohort['config'],
            'pair_indices': cohort['pair_indices'],
            'methods': cohort['methods'],
        }
        for seed, cohort in cohorts.items()
    },
    'summary': rows,
    'figures': [str(robustness_figure), str(feature_figure)],
}
result_path = RESULTS_DIR / 'robustness_evaluation_results.json'
result_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
print(f'Saved robustness results: {result_path}')
print(f'Saved cohort table: {csv_path}')

In [ ]:
def interval_text(values, seed):
    mean, lower, upper = bootstrap_mean_interval(values, seed=seed)
    return f'{mean:.3f} [{lower:.3f}, {upper:.3f}]'

rf_by_seed = {
    seed: cohorts[seed]['methods']['rectified_flow']['metrics']['sample_metrics']
    for seed in COHORT_SEEDS
}
paired_rows = []
for seed in COHORT_SEEDS:
    rf_samples = rf_by_seed[seed]
    rf_progress = np.asarray([
        sample['target_confidence'][-1] - sample['target_confidence'][0]
        for sample in rf_samples
    ])
    rf_monotonic = np.asarray([
        sample['target_confidence_monotonic'] for sample in rf_samples
    ], dtype=float)
    rf_feature_samples = cohorts[seed]['methods']['rectified_flow'][
        'feature_metrics'
    ]['sample_metrics']
    for method in METHOD_ORDER[1:]:
        samples = cohorts[seed]['methods'][method]['metrics']['sample_metrics']
        progress = np.asarray([
            sample['target_confidence'][-1] - sample['target_confidence'][0]
            for sample in samples
        ])
        monotonic = np.asarray([
            sample['target_confidence_monotonic'] for sample in samples
        ], dtype=float)
        feature_samples = cohorts[seed]['methods'][method][
            'feature_metrics'
        ]['sample_metrics']
        paired_rows.append({
            'seed': seed,
            'baseline': method,
            'paired target progression difference vs RF': interval_text(
                progress - rf_progress, seed
            ),
            'paired monotonic-path difference vs RF (pp)': interval_text(
                100 * (monotonic - rf_monotonic), seed + 1
            ),
            'paired target-feature distance Δ vs RF': interval_text(
                np.asarray([
                    s['target_reference_distance'][-1]
                    - s['target_reference_distance'][0]
                    for s in feature_samples
                ])
                - np.asarray([
                    s['target_reference_distance'][-1]
                    - s['target_reference_distance'][0]
                    for s in rf_feature_samples
                ]),
                seed + 2,
            ),
            'paired source-feature similarity at end vs RF': interval_text(
                np.asarray([s['source_feature_similarity'][-1] for s in feature_samples])
                - np.asarray([s['source_feature_similarity'][-1] for s in rf_feature_samples]),
                seed + 3,
            ),
        })

paired_header = '| Seed | Baseline | Paired target progression difference vs RF | Paired monotonic-path difference vs RF (pp) | Paired target-feature distance Δ vs RF | Paired source-feature similarity at end vs RF |\n|---:|---|---:|---:|---:|---:|\n'
paired_body = ''.join(
    f"| {row['seed']} | {row['baseline']} | {row['paired target progression difference vs RF']} | {row['paired monotonic-path difference vs RF (pp)']} | "
    f"{row['paired target-feature distance Δ vs RF']} | {row['paired source-feature similarity at end vs RF']} |\n"
    for row in paired_rows
)
display(Markdown(
    paired_header + paired_body
    + '\nPositive target-progression, monotonicity, and source-similarity differences favor the baseline; '
    + 'a negative target-feature-distance change difference means the baseline moved closer to target references. '
    + 'Intervals are paired within each test cohort.'
))
paired_csv = RESULTS_DIR / 'robustness_paired_differences.csv'
with paired_csv.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=paired_rows[0].keys())
    writer.writeheader()
    writer.writerows(paired_rows)
result['paired_comparisons'] = paired_rows
result_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
print(f'Saved paired robustness comparisons: {paired_csv}')